#### Con random forest

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score, r2_score

In [48]:
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.375, -60.0, -44.875]

In [49]:
fishing_ds = xr.open_dataset("../data/processed/dynamic/presence_SQA.nc")
fishing = fishing_ds["presence"]
fishing = fishing.fillna(0)

temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)

mask_ds = xr.open_dataset("../data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]
mask = mask.broadcast_like(temp)

month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp) #se añade como dinámica porque ya se ha corregido la forma

temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so = xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, join="inner")

cropped = lambda da: da.sel(
    lon=slice(min_lon, max_lon),
    lat=slice(min_lat, max_lat)
)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
fishing = cropped(fishing)
mask = cropped(mask)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)

month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)

In [50]:
# Target
y = fishing

# Feature stack
X = xr.Dataset({
    "temp": temp,
    "temp_bottom": temp_bottom,
    "chl": chl,
    "mixed": mixed,
    "depth": depth,
    "zo": zo,
    "so": so,
    "mask": mask,
    "month_sin": month_sin,
    "month_cos": month_cos,
    "lat": lat,
    "lon": lon
})

# Align everything (already done, but safe)
X, y = xr.align(X, y, join="inner")

# Stack spatial + temporal dimensions into rows
X_stacked = X.to_array().transpose("time", "lat", "lon", "variable").stack(samples=("time", "lat", "lon"))
y_stacked = y.stack(samples=("time", "lat", "lon"))

X_df = X_stacked.transpose("samples", "variable").to_pandas()
y_arr = y_stacked.values

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_arr, test_size=0.2, shuffle=False, random_state=42
)

In [51]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("R2:", r2_score(y_test, y_prob))

imp = pd.Series(rf.feature_importances_, index=X_df.columns)
imp.sort_values(ascending=False)

              precision    recall  f1-score   support

           0       0.94      0.99      0.97      4548
           1       0.52      0.12      0.20       324

    accuracy                           0.93      4872
   macro avg       0.73      0.56      0.58      4872
weighted avg       0.91      0.93      0.91      4872

PR-AUC: 0.4076965558899285
R2: 0.2019442134271474


variable
month_sin      0.178614
temp           0.178052
depth          0.138682
mixed          0.104130
chl            0.090250
temp_bottom    0.086355
so             0.076490
zo             0.070210
month_cos      0.050738
mask           0.026478
dtype: float64